# SpikeDistill-RUL official test comparison

This notebook reads frozen predictions from the completed baseline and student runs. A CPU runtime is sufficient. Run all cells in order and authorize Drive access when prompted.

**Primary benchmark.** NASA C-MAPSS FD001–FD004, raw official remaining useful life (RUL), one prediction per test engine. Checkpoints were selected on capped validation RMSE. Test labels are used here for scoring and diagnostics.

Each table entry summarizes seeds 42, 43, and 44 using the mean and sample standard deviation. The constant mean predictor is deterministic. The four subsets are reported separately.

**RAE** is the sum of absolute errors divided by the sum of absolute deviations of the true labels from their mean, expressed as a percentage. A value of 100% matches the test-mean reference. That reference is used to define the metric.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/ML projec")
raw = project / "CMaps"
runs = project / "SpikeDistill-RUL" / "colab_runs"
output = runs / "test_comparison_v1"
subsets = ["FD001", "FD002", "FD003", "FD004"]
seeds = [42, 43, 44]
sources = {
    "baselines_v1": {
        "mean": "Mean predictor",
        "deepnn": "DeepNN",
        "widenn": "WideNN",
        "rnn": "RNN",
        "lstm": "LSTM",
        "gru": "GRU",
        "bilstm": "BiLSTM",
        "cnn_lstm": "CNN-LSTM",
        "attention_bilstm": "Attention BiLSTM teacher",
        "small_lstm_kd": "Output-distilled small LSTM",
        "snn": "Fixed recurrent LIF SNN",
        "ann_to_snn": "ANN-to-SNN conversion",
    },
    "spikedistill_v1": {
        "learnable_snn": "Learnable SNN",
        "spike_output_kd": "Output-only spiking KD",
        "spike_temporal_kd": "Temporal spiking KD",
        "spikedistill": "SpikeDistill-RUL",
    },
}
labels = {key: label for group in sources.values() for key, label in group.items()}
metric_names = ["MAE", "MSE", "RMSE", "RAE_pct", "R2", "cmapss_score", "bias_cycles"]


## Validate the benchmark and recompute metrics

The official RUL file is ordered by engine ID. Each prediction must match the correct test engine, its last observed cycle, and the raw RUL label. The notebook also checks capped labels, all 192 expected model–subset–seed runs, shared prepared-data fingerprints, and agreement with the previously saved metrics.


In [ ]:
def official_endpoints(subset):
    rows = pd.read_csv(raw / f"test_{subset}.txt", sep=r"\s+", header=None, usecols=[0, 1])
    endpoints = rows.groupby(0, sort=True)[1].max()
    y = np.loadtxt(raw / f"RUL_{subset}.txt", ndmin=1)
    np.testing.assert_array_equal(endpoints.index.to_numpy(), np.arange(1, len(y) + 1))
    return pd.DataFrame({
        "engine_id": endpoints.index.to_numpy(),
        "end_cycle": endpoints.to_numpy(),
        "target_raw_cycles": y,
    })

def metrics(y, prediction):
    y, prediction = np.asarray(y, dtype=np.float64), np.asarray(prediction, dtype=np.float64)
    if not np.isfinite(y).all() or not np.isfinite(prediction).all():
        raise ValueError("A target or prediction is nonfinite")
    error = prediction - y
    deviation = y - y.mean()
    mse = np.mean(error ** 2)
    rae_denominator = np.sum(np.abs(deviation))
    r2_denominator = np.sum(deviation ** 2)
    return {
        "MAE": np.mean(np.abs(error)),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "RAE_pct": 100 * np.sum(np.abs(error)) / rae_denominator if rae_denominator else np.nan,
        "R2": 1 - np.sum(error ** 2) / r2_denominator if r2_denominator else np.nan,
        "cmapss_score": np.sum(np.expm1(np.where(error < 0, -error / 13, error / 10))),
        "bias_cycles": np.mean(error),
    }

def verify_saved_metrics(calculated, saved, n, context):
    if saved["n"] != n:
        raise ValueError(f"Saved test count mismatch: {context}")
    for name in ("MAE", "MSE", "RMSE", "R2", "cmapss_score"):
        expected = np.nan if saved[name] is None else saved[name]
        np.testing.assert_allclose(calculated[name], expected, rtol=1e-7, atol=1e-7,
                                   equal_nan=True, err_msg=f"{context}, {name}")

def verify_prepared_data(config, subset, fingerprints):
    if "source_sha256" in config:
        current = tuple(config["source_sha256"][f"{split}.npz"] for split in ("train", "validation"))
    else:
        current = (config["train_sha256"], config["validation_sha256"])
    if current != fingerprints.setdefault(subset, current):
        raise ValueError(f"Prepared training or validation data mismatch for {subset}")

expected = {subset: official_endpoints(subset) for subset in subsets}
records, prediction_frames, fingerprints = [], [], {}
for source, models in sources.items():
    for subset, benchmark in expected.items():
        for seed in seeds:
            for model, label in models.items():
                folder = runs / source / subset / f"seed_{seed}" / model
                path = folder / "test_predictions.csv"
                if not path.is_file():
                    raise FileNotFoundError(f"Missing official test predictions: {path}")
                frame = pd.read_csv(path).sort_values("engine_id").reset_index(drop=True)
                saved = json.loads((folder / "test_metrics.json").read_text())
                config = saved["config"]
                context = f"{subset}/{seed}/{model}"
                if (config["subset"], config["seed"], config["model"], saved["split"],
                        saved["prediction_units"]) != (subset, seed, model, "test", "cycles"):
                    raise ValueError(f"Wrong model, seed, subset, split, or prediction units: {context}")
                verify_prepared_data(config, subset, fingerprints)
                for col in benchmark.columns:
                    np.testing.assert_array_equal(frame[col].to_numpy(), benchmark[col].to_numpy(),
                                                  err_msg=f"{context}, {col}")
                cap = float(config["cap"])
                if cap != 125:
                    raise ValueError(f"Unexpected RUL cap in {context}: {cap}")
                np.testing.assert_array_equal(frame.target_capped_cycles,
                                              np.minimum(frame.target_raw_cycles, cap))
                prediction = frame.prediction_cycles.to_numpy()
                for protocol in ("raw", "capped"):
                    values = metrics(frame[f"target_{protocol}_cycles"], prediction)
                    verify_saved_metrics(values, saved[protocol], len(frame), f"{context}/{protocol}")
                    records.append(dict(subset=subset, model=model, label=label, seed=seed,
                                        target=protocol, engines=len(frame),
                                        selected_epoch=saved["selected_epoch"],
                                        parameters=saved["parameters"], **values))
                prediction_frames.append(frame.assign(subset=subset, model=model, seed=seed))
    print(f"Verified {source}: {len(models) * len(subsets) * len(seeds)} test runs")

per_seed = pd.DataFrame(records)
predictions = pd.concat(prediction_frames, ignore_index=True)
assert not per_seed.duplicated(["subset", "model", "seed", "target"]).any()
assert len(per_seed) == len(labels) * len(subsets) * len(seeds) * 2
group_keys = ["target", "subset", "model", "label"]
summary = per_seed.groupby(group_keys)[metric_names].agg(["mean", "std"]).reset_index()
summary.columns = [f"{name}_{'sample_sd' if stat == 'std' else stat}" if stat else name
                   for name, stat in summary.columns]
raw_summary = summary[summary.target == "raw"].copy()
output.mkdir(parents=True, exist_ok=True)
per_seed.to_csv(output / "test_metrics_per_seed.csv", index=False)
summary.to_csv(output / "test_metrics_mean_sd.csv", index=False)
predictions.to_csv(output / "test_predictions_verified.csv", index=False)
print(f"Verified {len(per_seed) // 2} runs on", {key: len(value) for key, value in expected.items()})
display(raw_summary.sort_values(["subset", "RMSE_mean"]).round(3))


## RMSE ranking for each subset

Bars show mean raw test RMSE. Whiskers show one sample standard deviation across the three seeds. The proposed student is highlighted. Lower RMSE is better. Read the ranking within each subset.


In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
colors = {"spikedistill": "#c45c26", "attention_bilstm": "#234979"}

def save_figure(fig, name):
    fig.savefig(output / f"{name}.png", dpi=180, bbox_inches="tight")
    fig.savefig(output / f"{name}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(15, 12), layout="constrained")
fig.suptitle("Official C-MAPSS test RMSE | mean ± sample SD across 3 seeds", fontsize=16)
for subset, ax in zip(subsets, axes.flat):
    rows = raw_summary[raw_summary.subset == subset].sort_values("RMSE_mean")
    ax.barh(rows.label, rows.RMSE_mean, xerr=rows.RMSE_sample_sd, capsize=3,
            color=[colors.get(model, "#8ab0ba") for model in rows.model])
    ax.invert_yaxis()
    ax.set(title=f"{subset} · {len(expected[subset])} engines", xlabel="RMSE (cycles)")
    ax.grid(axis="x", alpha=0.2)
save_figure(fig, "01_test_rmse")


## Relative absolute error across subsets

Lower values are better. Cells show mean RAE across the three seeds. Exact scores and seed variation are in the exported table.


In [ ]:
rae = raw_summary.pivot(index="model", columns="subset", values="RAE_pct_mean").reindex(
    index=list(labels), columns=subsets)
fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")
heat = ax.imshow(rae, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(subsets)), subsets)
ax.set_yticks(range(len(rae.index)), [labels[key] for key in rae.index])
ax.set_title("Official test RAE (%) | lower is better")
for i, row in enumerate(rae.to_numpy()):
    for j, value in enumerate(row):
        if np.isfinite(value):
            ax.text(j, i, f"{value:.1f}", ha="center", va="center",
                    color="white" if heat.norm(value) > 0.6 else "#202020")
fig.colorbar(heat, ax=ax, label="Mean RAE (%)")
save_figure(fig, "02_test_rae")


## What each student variant adds

The four variants share the same student architecture. Temporal KD adds hidden-state and attention matching together. SpikeDistill-RUL adds the firing-rate penalty to temporal KD. The plot shows prediction error for those comparisons. Training spike rates are saved separately in the epoch histories.


In [ ]:
students = list(sources["spikedistill_v1"])
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, layout="constrained")
fig.suptitle("Matched student variants on the official test engines", fontsize=15)
for subset, ax in zip(subsets, axes.flat):
    rows = raw_summary[raw_summary.subset == subset].set_index("model").loc[students]
    ax.errorbar(range(len(students)), rows.RMSE_mean, yerr=rows.RMSE_sample_sd,
                marker="o", color="#234979", capsize=4)
    ax.set(title=subset, ylabel="RMSE (cycles)")
    ax.set_xticks(range(len(students)), ["Supervised", "Output KD", "Temporal KD", "With sparsity"], rotation=12)
    ax.grid(axis="y", alpha=0.25)
save_figure(fig, "03_student_ablation")


## Where the student over- or under-predicts

Each point represents one official test engine. The plots use seed 42 for the teacher and student. Positive residuals indicate optimistic predictions, meaning the model estimated more life than the engine actually had.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
fig.suptitle("Teacher and student test residuals | seed 42", fontsize=15)
for subset, ax in zip(subsets, axes.flat):
    for model, marker in [("attention_bilstm", "o"), ("spikedistill", "^")]:
        rows = predictions[(predictions.subset == subset) & (predictions.seed == 42)
                           & (predictions.model == model)]
        ax.scatter(rows.target_raw_cycles, rows.prediction_cycles - rows.target_raw_cycles,
                   s=16, alpha=0.55, marker=marker, color=colors[model], label=labels[model])
    ax.axhline(0, color="#444444", linewidth=1)
    ax.set(title=subset, xlabel="True remaining life (cycles)", ylabel="Prediction − truth (cycles)")
    ax.legend(fontsize=8)
save_figure(fig, "04_test_residuals")
print("Tables and plots saved to", output)
